# Test Bottom-Up Tree Construction

This notebook exercises the bottom-up semantic tree builder in `build_llm_bottom_up_tree.py`.

What it does:
- loads a corpus that already contains embeddings
- builds leaf nodes
- asks an LLM to summarize clusters while building parent nodes bottom-up
- inspects the resulting tree
- optionally saves the tree to `trees/`

Before running the build cells, make sure the required API key for your selected LLM backend is available in the environment.


In [17]:
from __future__ import annotations

import json
import pickle
import sys
from pathlib import Path
from types import SimpleNamespace


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    while current != current.parent:
        if (current / "src").exists() and (current / "trees").exists():
            return current
        current = current.parent
    raise RuntimeError("Could not find the repository root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from tree_construction.build_llm_bottom_up_tree import (
    ClusterSummarizer,
    build_bottom_up_tree,
    build_leaf_nodes,
    count_leaves,
    count_nodes,
    load_source_records,
    max_branching,
)
from utils import setup_logger

REPO_ROOT


WindowsPath('C:/Users/jmgil/Desktop/TESE/LATTICE/llm-guided-retrieval')

In [18]:
# Adjust these values for the corpus and model you want to test.
args = SimpleNamespace(
    input=str(REPO_ROOT / "src" / "tree_construction" / "EU_conventions_example" /"Convention_ENG_chunks.json"),
    dataset="EU",
    subset="Convention_ENG",
    id_field="chunk_id",
    text_field="text",
    embedding_field="embedding",
    max_leaves=None,
    max_children=6,
    cluster_linkage="average",
    prompt_child_char_limit=700,
    summary_cache=str(REPO_ROOT / "results" / "tree_construction" / "Convention_ENG_cache.json"),
    llm_api_backend="openai",  # one of: openai, genai, vllm
    llm="gpt-4.1",
    llm_max_concurrent_calls=4,
    llm_api_timeout=120,
    llm_api_max_retries=4,
    llm_api_staggering_delay=0.05,
    vllm_base_url="http://localhost:8000/v1",
)

LOG_DIR = REPO_ROOT / "results" / "tree_construction"
LOG_DIR.mkdir(parents=True, exist_ok=True)
logger = setup_logger("bottom_up_tree_notebook", str(LOG_DIR / "bottom_up_tree_notebook.log"))

args


Log file already exists: C:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval\results\tree_construction\bottom_up_tree_notebook.log, appending to it.


namespace(input='C:\\Users\\jmgil\\Desktop\\TESE\\LATTICE\\llm-guided-retrieval\\src\\tree_construction\\EU_conventions_example\\Convention_ENG_chunks.json',
          dataset='EU',
          subset='Convention_ENG',
          id_field='chunk_id',
          text_field='text',
          embedding_field='embedding',
          max_leaves=None,
          max_children=6,
          cluster_linkage='average',
          prompt_child_char_limit=700,
          summary_cache='C:\\Users\\jmgil\\Desktop\\TESE\\LATTICE\\llm-guided-retrieval\\results\\tree_construction\\Convention_ENG_cache.json',
          llm_api_backend='openai',
          llm='gpt-4.1',
          llm_max_concurrent_calls=4,
          llm_api_timeout=120,
          llm_api_max_retries=4,
          llm_api_staggering_delay=0.05,
          vllm_base_url='http://localhost:8000/v1')

In [19]:
records = load_source_records(args, logger)
leaf_nodes = build_leaf_nodes(records, args, logger)

print(f"Loaded {len(records)} raw records")
print(f"Prepared {len(leaf_nodes)} leaf nodes")
print("First leaf preview:")
print(leaf_nodes[0].id)
print(leaf_nodes[0].desc[:400])


2026-05-07 03:55:58,780 - bottom_up_tree_notebook - INFO - Loading records from C:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval\src\tree_construction\EU_conventions_example\Convention_ENG_chunks.json
2026-05-07 03:55:59,535 - bottom_up_tree_notebook - INFO - Prepared 361 leaf nodes


Loaded 361 raw records
Prepared 361 leaf nodes
First leaf preview:
d26ae0e02e5722a6be2295b9065740487db48358
EU Preamble Convention for the Protection of Human Rights and Fundamental Freedoms Rome, 4.XI.1950 The GovernmenTs siGnaTory hereTo, being members of the Council of Europe, Considering the Universal Declaration of Human Rights proclaimed by the General Assembly of the United Nations on 10th December 1948; Considering that this Declaration aims at securing the universal and effective recognition an


In [20]:
summarizer = ClusterSummarizer(args, logger)
root = build_bottom_up_tree(leaf_nodes, args, summarizer, logger)
tree_dict = root.to_tree_dict()

print("Root description:")
print(tree_dict["desc"])
print()
print(f"Total nodes: {count_nodes(tree_dict)}")
print(f"Total leaves: {count_leaves(tree_dict)}")
print(f"Max branching: {max_branching(tree_dict)}")


2026-05-07 03:55:59,566 - bottom_up_tree_notebook - INFO - Initialized client for model: gpt-4.1
2026-05-07 03:56:00,013 - bottom_up_tree_notebook - INFO - Initialized OpenAI Responses client with model: gpt-4.1
2026-05-07 03:56:00,014 - bottom_up_tree_notebook - INFO - Building parent level 0 from 361 nodes
2026-05-07 03:56:00,172 - bottom_up_tree_notebook - INFO - Level 0 produced 106 parent nodes (max fanout 6)
2026-05-07 03:56:00,173 - bottom_up_tree_notebook - INFO - Building parent level 1 from 106 nodes
2026-05-07 03:56:00,212 - bottom_up_tree_notebook - INFO - Level 1 produced 40 parent nodes (max fanout 6)
2026-05-07 03:56:00,214 - bottom_up_tree_notebook - INFO - Building parent level 2 from 40 nodes
2026-05-07 03:56:00,234 - bottom_up_tree_notebook - INFO - Level 2 produced 22 parent nodes (max fanout 5)
2026-05-07 03:56:00,235 - bottom_up_tree_notebook - INFO - Building parent level 3 from 22 nodes
2026-05-07 03:56:00,250 - bottom_up_tree_notebook - INFO - Level 3 produced 

Root description:
ROOT Node: This subtree addresses questions about the foundational principles, legal procedures, and organizational structure of the European Union and the European Convention on Human Rights (ECHR). It covers the ECHR's objectives and relationship to broader human rights frameworks, the process for signing EU treaties, the structure and governance of EU institutions, and the rights, procedures, and committees established by EU Protocols. Use this subtree for queries on EU treaty signing, institutional roles, legislative processes, protected rights and freedoms, emergency derogations, committee mechanisms, and family law protections under EU law.. Key topics: ECHR foundational principles; EU treaty signing procedures; EU institutions and governance; EU legislative process; EU Protocol rights and freedoms; emergency derogations; committee structures; family law protections. 3 direct children. 361 leaf passages

Total nodes: 556
Total leaves: 361
Max branching: 6


In [21]:
# Inspect the first level of the tree to see what summaries the LLM produced.
for idx, child in enumerate(tree_dict.get("child") or []):
    print(f"[{idx}] {child['id']}")
    print(child["desc"][:800])
    print("-" * 120)


[0] level:5:cluster:0000:6883560925
ECHR Principles and EU Treaty Signing. Provides information on the foundational principles and preamble of the European Convention on Human Rights (ECHR), as well as the formal legal procedures for signing European Union treaties. Useful for questions about the ECHR's objectives, its relationship to broader human rights frameworks, and the step-by-step process and requirements for EU treaty authorization and signing.. Key topics: ECHR preamble; foundational principles; Council of Europe; Universal Declaration of Human Rights; EU treaty signing procedures; signatory authorization; legal requirements; official signatures. 2 direct children. 2 leaf passages
------------------------------------------------------------------------------------------------------------------------
[1] level:5:cluster:0001:2272cb4d5a
European Union Organization and Governance. Provides information on the structure, governance, and legislative processes of the European Union, 

In [22]:
# Render the constructed tree with Graphviz.
# If the diagram is too dense, lower MAX_DEPTH or LABEL_CHARS.
from graphviz import Digraph


MAX_DEPTH = None
LABEL_CHARS = 120
GRAPH_DIR = REPO_ROOT / "src" / "tree_construction" / "EU_conventions_example"
GRAPH_DIR.mkdir(parents=True, exist_ok=True)
GRAPH_PATH = GRAPH_DIR / "semantic_tree_graphviz"


def shorten(text, limit=LABEL_CHARS):
    text = str(text).replace("\n", " ").strip()
    return text if len(text) <= limit else text[: limit - 3] + "..."


def add_tree_to_dot(dot, node, path=(), max_depth=None):
    node_name = "root" if path == () else "node_" + "_".join(map(str, path))
    children = node.get("child") or []
    desc = shorten(node.get("desc", ""))
    node_id = node.get("id")
    is_truncated = max_depth is not None and len(path) >= max_depth and len(children) > 0
    truncation_note = "\n[TRUNCATED]" if is_truncated else ""
    label = f"{node_name}\n{node_id}{truncation_note}\n{desc}" if node_id is not None else f"{node_name}{truncation_note}\n{desc}"

    fillcolor = "lightsalmon" if is_truncated else "lightgoldenrod1"
    dot.node(node_name, label=label, shape="box", style="rounded,filled", fillcolor=fillcolor)

    if max_depth is not None and len(path) >= max_depth:
        return

    for idx, child in enumerate(children):
        child_path = path + (idx,)
        child_name = "node_" + "_".join(map(str, child_path))
        add_tree_to_dot(dot, child, child_path, max_depth=max_depth)
        dot.edge(node_name, child_name)


dot = Digraph(comment="Semantic Tree")
dot.attr(rankdir="TB")
dot.attr("node", fontname="Helvetica", fontsize="10")
dot.attr("edge", color="gray50")

add_tree_to_dot(dot, tree_dict, max_depth=MAX_DEPTH)
png_path = dot.render(str(GRAPH_PATH), format="svg", cleanup=True)
print(f"Saved Graphviz PNG to {png_path}")
dot


Saved Graphviz PNG to C:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval\src\tree_construction\EU_conventions_example\semantic_tree_graphviz.svg


In [23]:
def count_rendered_nodes(node, path=(), max_depth=None):
    total = 1
    if max_depth is not None and len(path) >= max_depth:
        return total
    for idx, child in enumerate(node.get("child") or []):
        total += count_rendered_nodes(child, path + (idx,), max_depth=max_depth)
    return total

def count_rendered_terminal_nodes(node, path=(), max_depth=None):
    children = node.get("child") or []
    if max_depth is not None and len(path) >= max_depth:
        return 1
    if not children:
        return 1
    total = 0
    for idx, child in enumerate(children):
        total += count_rendered_terminal_nodes(child, path + (idx,), max_depth=max_depth)
    return total

print("Rendered nodes:", count_rendered_nodes(tree_dict, max_depth=MAX_DEPTH))
print("Full nodes:", count_nodes(tree_dict))
print("Rendered terminal nodes:", count_rendered_terminal_nodes(tree_dict, max_depth=MAX_DEPTH))
print("Full leaves:", count_leaves(tree_dict))
if MAX_DEPTH is not None:
    print("Note: rendered terminal nodes may be higher than true leaves because truncated internal nodes are drawn as endpoints.")

Rendered nodes: 556
Full nodes: 556
Rendered terminal nodes: 361
Full leaves: 361


In [24]:
# Optional: persist the notebook-built tree so it can be used by the retrieval pipeline.
OUTPUT_DIR = REPO_ROOT / "trees" / args.dataset / "eu_conventions_notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pickle_path = OUTPUT_DIR / "eu_conventions_tree-bottom-up-llm.pkl"
json_path = OUTPUT_DIR / "eu_conventions_tree-bottom-up-llm.json"

pickle_path.write_bytes(pickle.dumps(tree_dict))
json_path.write_text(json.dumps(tree_dict, ensure_ascii=False, indent=2), encoding="utf-8")

print(pickle_path)
print(json_path)


C:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval\trees\EU\eu_conventions_notebook\eu_conventions_tree-bottom-up-llm.pkl
C:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval\trees\EU\eu_conventions_notebook\eu_conventions_tree-bottom-up-llm.json
